# Capstone — Which pages should an editor review first?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadSulaiman08/FLYRANK-ML-INTERNSHIP/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane 2 · Refresh / Content Opportunity Scoring** · Muhammad Sulaiman · FlyRank ML Internship · November 2026

This notebook is the evidence behind my deployed research paper (link in `submission/paper_url.txt`). Every number in the paper is printed by a cell below.

**Abstract.** An editor can review only a few pages a week, so the decision this work supports is *the order* of that review, measured by Precision@50. On 30,000 pseudonymized pages from 32 clients, I compared two transparent rules with four learned models on the same five client-grouped folds (every test fold holds clients the model never saw). The learned models ranked the review queue clearly better than both rules (Precision@50 ≈ 0.77–0.78 vs 0.60 and 0.46; random order 0.54), but they were statistically indistinguishable from each other, so complexity did not earn extra credit. A stricter check on the full warehouse with a true future-window label (first half of March 2026 → second half) found a much weaker honest signal (ROC AUC ≈ 0.60), which is the main limitation. The output is a ranked review queue with reason codes and suggested actions — decision support, not a causal claim.

> Run order: **Runtime → Run all**. Section 4b and parts of Section 2 read the gated warehouse and need a Colab Secret named `HF_TOKEN` (never paste the token in a cell — this repo is public).

## 1. Question

**Research question.** Using only signals known before the decision, can a learned ranking put declining pages at the top of an editor's review queue more reliably than transparent rules — on clients it has never seen?

**Decision supported.** Which pages a content editor reviews first, given limited hours. **Unit:** one page (one `content_id`). **Output:** a ranked queue with a score, reason codes and a suggested action. **Metric (fixed before training):** Precision@50 — of the 50 pages put first, the share that are actually declining — plus P@20, P@100, ROC AUC and average precision. **Cost of a wrong call:** a false alarm wastes editor time; a miss lets a valuable page keep losing traffic.

In [ ]:
import os, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

CSV = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(CSV):
    for up in ("..", "../.."):
        if os.path.exists(os.path.join(up, CSV)):
            os.chdir(up); break
    else:
        if not os.path.isdir("starter"):
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/flyrank-bih/flyrank-ml-internship-starter", "starter"], check=True)
        os.chdir("starter")

import sklearn
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__, "| seed", RANDOM_STATE)

df = pd.read_csv(CSV)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id").reset_index(drop=True)
df["y"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{len(df):,} pages | {df['client_id'].nunique()} clients | declining (label=1): {df['y'].mean():.3f}")

## 2. Data

**Primary — starter release** (`data/raw/content_refresh_anonymized.csv`): 30,000 pseudonymized pages from 32 clients, trailing-90-day metrics, one row per page (grain checked below: every `content_id` unique). Rate columns are ×100 percentages (`ctr = 0.5` means 0.5%).

**Validation — warehouse release** (`FlyRank/internship-warehouse` on Hugging Face, build v20260703): `fact_content_daily_performance`, mid-panel month **March 2026** only (not the `_sample`, which is the final month and stays sealed), plus `dim_clients` for history depth.

**Labels.** Starter: `trend_direction == "down"` — a *current-trend proxy*. Warehouse: a true *future-window* label — daily impressions in Mar 16–31 fell more than 20% below the Mar 1–15 daily average (pages with ≥ 50 impressions in Mar 1–15).

**Excluded, and why:** `trend_direction` / `trend_pct` (the label is built from them — leakage); `content_id` / `client_id` (pseudonyms — grouping and splitting only); any Mar 16–31 value as a warehouse feature (future at the decision moment); GA4 session columns on the warehouse (available on only ~4% of March rows, zero-filled elsewhere). No client names, domains, URLs, queries or titles exist in either release, and none appear here.

In [ ]:
print("Starter grain: rows =", len(df), "| unique content_id =", df["content_id"].nunique())
print(df["trend_direction"].value_counts().to_string())

# --- Warehouse connection (token from Colab Secrets, never typed in a cell) ---
%pip -q install duckdb huggingface_hub
import getpass, duckdb
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token: ")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

wh = con.sql(f"""SELECT COUNT(*) AS rows, COUNT(DISTINCT client_hash_id) AS clients,
                        COUNT(DISTINCT content_hash_id) AS pages, MIN(report_date) AS first_day,
                        MAX(report_date) AS last_day,
                        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_ga4_available
                 FROM {MONTH}""").df()
hist = con.sql(f"""SELECT COUNT(*) AS clients,
                          COUNT(*) FILTER (WHERE gsc_data_start <= DATE '2025-03-01') AS clients_12m_plus,
                          COUNT(*) FILTER (WHERE gsc_data_start IS NULL) AS clients_no_gsc_start
                   FROM {CLIENTS}""").df()
print("\nWarehouse slice (March 2026):"); print(wh.to_string(index=False))
print("\nClient history depth (dim_clients):"); print(hist.to_string(index=False))

## 3. Methodology

**Assumptions.** A page "trending down" now is a reasonable stand-in for "worth reviewing" (starter proxy); the warehouse future-window label tests whether that holds forward in time. **Features (starter, 35 after one-hot):** search volume, competition, CPC, word/char count, log impressions/clicks/sessions/AI sessions (90d), days with impressions/sessions, content age, days since update, CTR, average position, engagement rate, scroll rate, AI traffic %, plus competition level, content type, intent and position tier. All are pre-decision observable signals; a code check asserts the label columns are absent.

**Baselines (no training):** (1) the hand rule *stale × visible* — not updated in 180+ days and 500+ impressions, ranked by impressions; (2) the reference pipeline's weighted rule (0.40 visibility + 0.30 freshness risk + 0.25 position opportunity + 0.05 depth gap). **Models:** Logistic Regression, depth-4 Decision Tree, Random Forest, Gradient Boosting (fixed seeds).

**Validation design.** 5-fold `GroupKFold` on `client_id`: every test fold contains only unseen clients ("shared clients: 0" printed below). Baselines and models are scored on the identical folds. **Leakage checks:** label columns asserted absent; on the warehouse I deliberately add a label-window column to show the score inflation, then remove it.

In [ ]:
from sklearn.model_selection import GroupKFold

for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df["log_" + c] = np.log1p(df[c].clip(lower=0))
NUM = ["search_volume", "competition", "cpc", "word_count", "char_count",
       "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
       "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
       "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
CAT = ["competition_level", "content_type", "main_intent", "position_tier"]
LEAKY = {"trend_direction", "trend_pct", "y"}
assert not LEAKY & set(NUM + CAT), "leakage: label-derived column in features"

X_num = df[NUM].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
X = pd.concat([X_num, pd.get_dummies(df[CAT].fillna("missing").astype(str)).astype(int)], axis=1)
y, groups = df["y"].values, df["client_id"].values
folds = list(GroupKFold(n_splits=5).split(X, y, groups))
print(f"{X.shape[1]} features | leakage guard passed")
for i, (tr, te) in enumerate(folds, 1):
    print(f"fold {i}: train {len(tr):>6,} pages / {len(set(groups[tr]))} clients | "
          f"test {len(te):>5,} pages / {len(set(groups[te]))} clients | shared clients: {len(set(groups[tr]) & set(groups[te]))}")

## 4. Results (vs baseline)

### 4a. Starter release — same folds, same metrics

The table is the mean over the five client-grouped folds; `P@50 std` shows how much a result swings between groups of clients. The fold-by-fold table below it counts how often each model beats the hand rule.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

def p_at_k(y_true, score, k):
    order = np.argsort(-np.asarray(score), kind="stable")
    return np.asarray(y_true)[order[:k]].mean()

def pct(s):
    return pd.Series(s).rank(pct=True).values

vis = pct(np.log1p(df["impressions_90d"]))
fresh = pct(df["days_since_last_update"])
pos = df["avg_position"].clip(1, 50)
pos_opp = (1 - (pos - pos.min()) / (pos.max() - pos.min())) * vis * (df["avg_position"] > 0)
depth = (1 - pct(df["word_count"].fillna(0))) * vis
BASELINES = {
    "Hand rule (stale x visible)": ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)) * df["impressions_90d"],
    "Weighted rule (pipeline baseline)": 0.40 * vis + 0.30 * fresh + 0.25 * pos_opp + 0.05 * depth,
}
MODELS = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "Decision Tree (depth 4)": DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=RANDOM_STATE),
}

rows, oof = [], {n: np.zeros(len(df)) for n in list(BASELINES) + list(MODELS)}
for f, (tr, te) in enumerate(folds):
    for name, s in BASELINES.items():
        oof[name][te] = np.asarray(s, dtype=float)[te]
    for name, m in MODELS.items():
        m.fit(X.iloc[tr], y[tr])
        oof[name][te] = m.predict_proba(X.iloc[te])[:, 1]   # out-of-fold: scored by a model that never saw this client
    for name in oof:
        sc = oof[name][te]
        rows.append({"method": name, "fold": f + 1, "P@20": p_at_k(y[te], sc, 20), "P@50": p_at_k(y[te], sc, 50),
                     "P@100": p_at_k(y[te], sc, 100), "ROC AUC": roc_auc_score(y[te], sc),
                     "Avg precision": average_precision_score(y[te], sc)})

res = pd.DataFrame(rows)
table = res.groupby("method", sort=False)[["P@20", "P@50", "P@100", "ROC AUC", "Avg precision"]].mean()
table["P@50 std"] = res.groupby("method", sort=False)["P@50"].std()
table.loc["Random order (base rate)"] = [y.mean(), y.mean(), y.mean(), 0.5, y.mean(), 0.0]
print("Mean over 5 client-grouped folds (each test fold = clients never seen in training):")
print(table.round(3).to_string())

by_fold = res.pivot(index="fold", columns="method", values="P@50")
print("\nP@50 per fold:")
print(by_fold[["Hand rule (stale x visible)", "Logistic Regression", "Random Forest", "Gradient Boosting"]].round(2).to_string())
for m in ["Logistic Regression", "Random Forest", "Gradient Boosting"]:
    wins = int((by_fold[m] > by_fold["Hand rule (stale x visible)"]).sum())
    print(f"{m}: beats the hand rule in {wins} of 5 folds")

### 4b. Warehouse — a true future-window label (March 2026)

Features from Mar 1–15 only (impressions, clicks, CTR, average position, days seen); label from Mar 16–31. The model is scored on held-out clients, first honestly, then with one label-window column added on purpose to show how leakage inflates the score.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_1_15,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks      ELSE 0 END) AS clk_1_15,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_1_15,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN report_date END) AS days_seen_1_15,
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_16_31
    FROM {MONTH}
    GROUP BY 1, 2
    HAVING imp_1_15 >= 50
    ORDER BY 1, 2
""").df()
frame["ctr_1_15"] = frame["clk_1_15"] / frame["imp_1_15"]
frame["declined"] = ((frame["imp_16_31"] / 16) < 0.8 * (frame["imp_1_15"] / 15)).astype(int)
frame["pos_1_15"] = frame["pos_1_15"].fillna(frame["pos_1_15"].median())
WH_FEATURES = ["imp_1_15", "clk_1_15", "ctr_1_15", "pos_1_15", "days_seen_1_15"]

wy, wg = frame["declined"].values, frame["client_hash_id"].values
wtr, wte = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE).split(frame, wy, wg))
def wh_auc(cols):
    m = DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, random_state=RANDOM_STATE)
    m.fit(frame.iloc[wtr][cols], wy[wtr])
    return roc_auc_score(wy[wte], m.predict_proba(frame.iloc[wte][cols])[:, 1])

wh_honest, wh_leaky = wh_auc(WH_FEATURES), wh_auc(WH_FEATURES + ["imp_16_31"])
print(f"{len(frame):,} pages (>= 50 impressions in Mar 1-15) from {frame['client_hash_id'].nunique()} clients | declined share: {wy.mean():.3f}")
print(f"Test: {len(wte):,} pages from {len(set(wg[wte]))} unseen clients")
print(f"Honest 5 features         ROC AUC: {wh_honest:.3f}")
print(f"+ imp_16_31 (deliberate leak) ROC AUC: {wh_leaky:.3f}  <- inflated by label-window information")
frame = frame.drop(columns=["imp_16_31"])
print("Leak removed. Honest number kept:", round(wh_honest, 3))

## 5. Limitations

What this work cannot claim — the code below backs the error analysis.

- **Proxy label.** The starter label is the *current* trend, not a future outcome; 4a says which pages *look like* declining pages now. With a true future label (4b) the honest signal is much weaker (ROC AUC ≈ 0.60).
- **No causal claim.** Nothing here shows that refreshing a page causes recovery, or reveals how Google ranks pages.
- **Models are not separable.** LR, RF and GB differ by less than their fold-to-fold spread (P@50 std ≈ 0.12–0.15); the honest finding is "learned beats rules", not "this model beats that one".
- **Thin, unbalanced history.** Only 3 of 104 warehouse clients have 12+ months of search history and 37 have no GSC start date, so seasonality cannot be checked; one starter fold is a single large client.
- **Low CTR is not proof of a bad title.** The queue is dominated by visible pages with almost no clicks; a search-page feature (answer box, AI overview) can take those clicks. The action is a *review*, not a rewrite order.
- **Engagement is out of scope on the warehouse:** GA4 is available on ~4% of March rows.

In [ ]:
from sklearn.inspection import permutation_importance

tr, te = folds[0]
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample",
                            n_jobs=-1, random_state=RANDOM_STATE).fit(X.iloc[tr], y[tr])
perm = permutation_importance(rf, X.iloc[te], y[te], scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("Top 8 features by permutation importance (drop in ROC AUC when shuffled, held-out clients):")
print(imp.head(8).round(4).to_string())

test = df.iloc[te].copy()
test["score"] = rf.predict_proba(X.iloc[te])[:, 1]
top = test.sort_values("score", ascending=False).head(50)
print(f"\nTop-50 on held-out fold 1: {top['y'].sum()} declining, {(1 - top['y']).sum()} false alarms")
print("False alarms' real trend:", top.loc[top["y"] == 0, "trend_direction"].value_counts().to_dict())
test["age_group"] = pd.cut(test["content_age_days"], [0, 180, 365, 730, 10**5], labels=["<6m", "6-12m", "1-2y", "2y+"])
print("\nShare declining and mean model score by content age (held-out fold 1):")
print(test.groupby("age_group", observed=True).agg(pages=("y", "size"), declining=("y", "mean"), mean_score=("score", "mean")).round(3).to_string())

## 6. Ranked recommendations

The queue uses the **out-of-fold Logistic Regression score** (each page scored by a model that never saw its client), restricted to pages with real demand (≥ 100 impressions in 90 days). Why Logistic Regression and not the model with the highest mean: the three learned models are within noise of each other on P@50 (0.768–0.784), but Logistic Regression is the only one that beat the hand rule in **all 5** folds, and it is the most readable. The cell prints the pooled queue precision for all three so the choice is transparent. Each page gets **reason codes** (transparent rules a human can check) and one **suggested action**:

| Reason code | Rule | Action |
|---|---|---|
| `low_ctr_visible` | ≥ 500 impressions, position 1–20, CTR < 0.5% | Review title, meta & SERP snippet (check search-page features first) |
| `page_one_decay_risk` | position ≤ 10, age ≥ 180 days | Protect: refresh facts, examples, date |
| `stale_visible` | not updated 180+ days, ≥ 500 impressions | Refresh content |
| `thin_visible` | < 1,200 words, ≥ 250 impressions | Expand content |
| `deep_position` | position > 20 | Review intent match & internal links |
| *(none)* | — | Monitor |

**Confidence:** *high* when score ≥ 0.70 and ≥ 500 impressions; otherwise *medium*.

**What the queue shows (observed):** the top of the queue is dominated by pages that are highly visible but get almost no clicks — the model's two largest coefficients are impressions (+) and clicks (−). So the main playbook move is a snippet/title review of high-impression, near-zero-click pages, followed by intent and internal-link review for pages slipping past position 20.

In [ ]:
q = df.copy()
q["score"] = oof["Logistic Regression"]
q = q[q["impressions_90d"] >= 100].copy()

def reasons(r):
    codes = []
    if r.impressions_90d >= 500 and 0 < r.avg_position <= 20 and r.ctr < 0.5: codes.append("low_ctr_visible")
    if 0 < r.avg_position <= 10 and r.content_age_days >= 180:               codes.append("page_one_decay_risk")
    if r.days_since_last_update >= 180 and r.impressions_90d >= 500:         codes.append("stale_visible")
    if 0 < (r.word_count if pd.notna(r.word_count) else 0) < 1200 and r.impressions_90d >= 250: codes.append("thin_visible")
    if r.avg_position > 20:                                                  codes.append("deep_position")
    return codes
ACTION = {"low_ctr_visible": "Review title, meta & SERP snippet", "page_one_decay_risk": "Protect: refresh facts & date",
          "stale_visible": "Refresh content", "thin_visible": "Expand content",
          "deep_position": "Review intent match & internal links"}

q = q.sort_values("score", ascending=False).reset_index(drop=True)
q["rank"] = q.index + 1
q["reason_codes"] = q.apply(reasons, axis=1)
q["action"] = q["reason_codes"].map(lambda c: ACTION[c[0]] if c else "Monitor")
q["confidence"] = np.where((q["score"] >= 0.70) & (q["impressions_90d"] >= 500), "high", "medium")

print(f"Eligible pages (>= 100 impressions): {len(q):,}")
for m in ["Logistic Regression", "Random Forest", "Gradient Boosting"]:
    s_ = df.assign(s=oof[m])[df["impressions_90d"] >= 100].sort_values("s", ascending=False)
    print(f"  {m:20} pooled queue: top 50 {s_['y'].head(50).mean():.1%} declining | top 200 {s_['y'].head(200).mean():.1%}")
lr = MODELS["Logistic Regression"].fit(X, y)
coef = pd.Series(lr[-1].coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print("\nLogistic Regression (fit on all pages) - strongest standardized coefficients (+ = more likely declining):")
print(coef.head(6).round(3).to_string())
for k in (50, 200):
    print(f"Queue top {k}: {q['y'].head(k).mean():.1%} actually declining (pooled out-of-fold scores)")
cols = ["rank", "score", "impressions_90d", "avg_position", "ctr", "content_age_days", "days_since_last_update", "confidence", "action"]
print("\nTop 20 of the review queue (IDs omitted):")
print(q[cols].head(20).round({"score": 3, "avg_position": 1, "ctr": 2}).to_string(index=False))
print("\nAction mix in the top 200:")
print(q["action"].head(200).value_counts().to_string())

## 7. Artifacts the paper embeds

The charts the deployed paper shows, regenerated from the results above (saved to `work/outputs/` when run locally).

In [ ]:
import matplotlib.pyplot as plt
os.makedirs("work/outputs", exist_ok=True)

order = ["Random order (base rate)", "Weighted rule (pipeline baseline)", "Hand rule (stale x visible)",
         "Decision Tree (depth 4)", "Gradient Boosting", "Logistic Regression", "Random Forest"]
t = table.loc[order]
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#bbbbbb", "#9aa7b8", "#9aa7b8", "#6f8fc9", "#2b5aa8", "#2b5aa8", "#2b5aa8"]
ax.barh(t.index, t["P@50"], xerr=t["P@50 std"], color=colors, capsize=3)
ax.set_xlabel("Precision@50 (mean ± std over 5 client-grouped folds)"); ax.set_xlim(0, 1)
ax.set_title("Learned rankings vs transparent rules")
plt.tight_layout(); plt.savefig("work/outputs/fig1_p50.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
imp.head(8)[::-1].plot.barh(ax=ax, color="#2b5aa8")
ax.set_xlabel("Drop in ROC AUC when shuffled (held-out clients)"); ax.set_title("What the model leans on")
plt.tight_layout(); plt.savefig("work/outputs/fig2_importance.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(8, 3))
q["action"].head(200).value_counts()[::-1].plot.barh(ax=ax, color="#2b5aa8")
ax.set_xlabel("Pages in the top 200 of the queue"); ax.set_title("Suggested actions")
plt.tight_layout(); plt.savefig("work/outputs/fig3_actions.png", dpi=150); plt.show()

summary = {"starter_table": table.round(3).to_dict(), "warehouse_honest_auc": round(wh_honest, 3),
           "warehouse_leaky_auc": round(wh_leaky, 3), "warehouse_pages": int(len(frame)),
           "queue_model": "Logistic Regression", "queue_top50_precision": round(float(q["y"].head(50).mean()), 3), "queue_top200_precision": round(float(q["y"].head(200).mean()), 3)}
print(summary)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.